# Comprehensive Deep Learning Masterclass (PyTorch)

Welcome to the Deep Learning Masterclass! This notebook explains the fundamental concepts of Deep Learning, Artificial Neural Networks (ANNs), and Convolutional Neural Networks (CNNs). We will use **PyTorch** to build and train these models.

### Table of Contents
1. [What is a Tensor?](#1.-What-is-a-Tensor?)
2. [Data Loading (Fashion MNIST)](#2.-Data-Loading-(Fashion-MNIST))
3. [Model 1: Multi-Layer Perceptron (MLP)](#3.-Model-1:-Multi-Layer-Perceptron-(MLP))
4. [Training Loop & Backpropagation](#4.-Training-Loop-&-Backpropagation)
5. [Model 2: Convolutional Neural Network (CNN)](#5.-Model-2:-Convolutional-Neural-Network-(CNN))
\n

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import matplotlib.pyplot as plt
import numpy as np

# Set device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
\n

## 1. What is a Tensor?
In PyTorch, everything is based on **Tensors**. A tensor is a mathematical object analogous to but more general than a vector (1D tensor) or matrix (2D tensor). An image with RGB channels is typically represented as a 3D tensor (Channels x Height x Width). Tensors can be easily moved to a GPU for massive parallel computation.\n

## 2. Data Loading (Fashion MNIST)
We will use the Fashion MNIST dataset, which contains 28x28 grayscale images of clothing items across 10 classes.\n

In [ ]:
# Define transforms to convert images to PyTorch Tensors and normalize them
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Download and load training data
trainset = torchvision.datasets.FashionMNIST(root='./data', train=True, download=True, transform=transform)
trainloader = torch.utils.data.DataLoader(trainset, batch_size=64, shuffle=True)

# Download and load test data
testset = torchvision.datasets.FashionMNIST(root='./data', train=False, download=True, transform=transform)
testloader = torch.utils.data.DataLoader(testset, batch_size=64, shuffle=False)

# Define classes
classes = ('T-shirt/top', 'Trouser', 'Pullover', 'Dress', 'Coat', 'Sandal', 'Shirt', 'Sneaker', 'Bag', 'Ankle boot')

# Function to show an image
def imshow(img):
    img = img / 2 + 0.5     # unnormalize
    npimg = img.numpy()
    plt.imshow(np.transpose(npimg, (1, 2, 0)), cmap='gray')
    plt.show()

# Get some random training images
dataiter = iter(trainloader)
images, labels = next(dataiter)

# Show images
imshow(torchvision.utils.make_grid(images[:4]))
print(' '.join(f'{classes[labels[j]]}' for j in range(4)))
\n

## 3. Model 1: Multi-Layer Perceptron (MLP)

### Theory & Intuition
An MLP is the simplest form of an Artificial Neural Network. It consists of an input layer, one or more hidden layers, and an output layer.
Every node in one layer connects to every node in the next layer (Fully Connected).

**Mathematics of a Layer:**
`y = ActivationFunction(W*x + b)`
Where `W` is the weight matrix and `b` is the bias. The activation function (like ReLU) introduces non-linearity, allowing the network to learn complex patterns.\n

In [ ]:
class MLP(nn.Module):
    def __init__(self):
        super(MLP, self).__init__()
        # 28x28 image = 784 input features
        self.fc1 = nn.Linear(28 * 28, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10) # 10 output classes

    def forward(self, x):
        # Flatten the image
        x = x.view(-1, 28 * 28)
        # Apply layers with ReLU activation
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.fc3(x) # No activation here, CrossEntropyLoss applies Softmax internally
        return x

mlp_model = MLP().to(device)
print(mlp_model)
\n

## 4. Training Loop & Backpropagation

To train a neural network, we need three things:
1. **Forward Pass**: Pass data through the network to get predictions.
2. **Loss Function**: Calculate how wrong the predictions are (e.g., Cross-Entropy Loss).
3. **Backward Pass (Backpropagation)**: Calculate the gradient of the loss with respect to every weight in the network.
4. **Optimizer step**: Update the weights slightly in the opposite direction of the gradient using Gradient Descent (or variants like Adam).\n

In [ ]:
# Define Loss and Optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(mlp_model.parameters(), lr=0.001)

# Training loop
epochs = 2 # Keeping it small for demonstration
for epoch in range(epochs):
    running_loss = 0.0
    for i, data in enumerate(trainloader, 0):
        inputs, labels = data[0].to(device), data[1].to(device)

        # Zero the parameter gradients
        optimizer.zero_grad()

        # Forward + backward + optimize
        outputs = mlp_model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item()
        if i % 300 == 299:    # print every 300 mini-batches
            print(f'[{epoch + 1}, {i + 1:5d}] loss: {running_loss / 300:.3f}')
            running_loss = 0.0

print('Finished Training MLP')
\n

## 5. Model 2: Convolutional Neural Network (CNN)

### Theory & Intuition
MLPs flatten images into 1D vectors, losing spatial information (like the shape of an edge).
CNNs preserve spatial structure by sliding **Filters (Kernels)** over the 2D image.
- **Convolution**: Applies the filter to extract features (edges, textures).
- **Max Pooling**: Downsamples the image, keeping the most important features and reducing computation.\n

In [ ]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN, self).__init__()
        # 1 input channel (grayscale), 16 output channels, 3x3 square convolution
        self.conv1 = nn.Conv2d(1, 16, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.conv2 = nn.Conv2d(16, 32, 3, padding=1)
        # After two max pools, the 28x28 image is 7x7
        self.fc1 = nn.Linear(32 * 7 * 7, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 32 * 7 * 7) # Flatten
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

cnn_model = CNN().to(device)
print(cnn_model)
\n